In [ ]:
!pip install -q transformers datasets sentence-transformers faiss-cpu tqdm

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
from tqdm import tqdm
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)

model = SentenceTransformer("intfloat/e5-large-v2", device=device)

batch_size = 256
all_embs = []

for i in tqdm(range(0, len(paragraphs), batch_size), desc="Embedding paragraphs"):
    batch = paragraphs[i:i+batch_size]
    embs = model.encode(
        batch,
        convert_to_numpy=True,
        show_progress_bar=False
    )
    all_embs.append(embs)

embeddings = np.vstack(all_embs)
print("Embeddings shape:", embeddings.shape)  # should be (29999, 768)


In [ ]:
import faiss
import numpy as np

# normalize for cosine search via inner product
faiss.normalize_L2(embeddings)

dim = embeddings.shape[1]  # should be 1024
index = faiss.IndexFlatIP(dim)  # inner product index
index.add(embeddings)

print("Index size:", index.ntotal)  # should be 29999


In [ ]:
import os
import json

SAVE_DIR = "/content/drive/MyDrive/rag_index_v2"
os.makedirs(SAVE_DIR, exist_ok=True)

with open(f"{SAVE_DIR}/paragraphs.json", "w", encoding="utf-8") as f:
    json.dump(paragraphs, f, ensure_ascii=False)

np.save(f"{SAVE_DIR}/wiki_emb.npy", embeddings)

faiss.write_index(index, f"{SAVE_DIR}/wiki_faiss.index")

print("Saved new RAG index to:", SAVE_DIR)


In [ ]:
import json
import os
from tqdm import tqdm
import numpy as np
import faiss
import torch
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

BASE = "/content/drive/MyDrive/rag_index_v2"
PARA_PATH = f"{BASE}/paragraphs.json"
EMB_PATH  = f"{BASE}/wiki_emb.npy"
INDEX_PATH = f"{BASE}/wiki_faiss.index"

with open(PARA_PATH, "r", encoding="utf-8") as f:
    paragraphs = json.load(f)

print("Loaded paragraphs:", len(paragraphs))

index = faiss.read_index(INDEX_PATH)
print("Loaded FAISS index size:", index.ntotal)

assert len(paragraphs) == index.ntotal, \
    "Paragraph count and FAISS index size mismatch! (Should not happen now)"

embed_model = SentenceTransformer("intfloat/e5-large-v2", device=device)

def retrieve_evidences(query, top_k=2):
    q = embed_model.encode(f"query: {query}", convert_to_numpy=True)
    faiss.normalize_L2(q.reshape(1, -1))
    scores, idx = index.search(q.reshape(1, -1), top_k)
    # idx[0] is a list of integer positions guaranteed < len(paragraphs)
    return [paragraphs[i] for i in idx[0]], scores[0]

nli_model_name = "facebook/bart-large-mnli"
nli_tokenizer = AutoTokenizer.from_pretrained(nli_model_name)
nli_model = AutoModelForSequenceClassification.from_pretrained(nli_model_name).to(device)
nli_model.eval()

def fast_nli(claim, evidence):
    inputs = nli_tokenizer(claim, evidence, return_tensors="pt", truncation=True).to(device)
    with torch.no_grad():
        logits = nli_model(**inputs).logits
        probs = F.softmax(logits, dim=-1)[0].cpu().numpy()
    return int(np.argmax(probs))  # 0=contradiction, 1=neutral, 2=entailment

CACHE_PATH = "/content/drive/MyDrive/ultrafb_truth_cache.json"

# load cache
truth_cache = {}
if os.path.exists(CACHE_PATH):
    with open(CACHE_PATH, "r", encoding="utf-8") as f:
        truth_cache = json.load(f)
    print("Loaded truth cache entries:", len(truth_cache))
else:
    print("No cache found, starting new one.")

import re

def split_sentences(text):
    sentences = re.split(r'[.!?]\s+', text.strip())
    sentences = [s.strip() for s in sentences if len(s.strip()) > 0]
    return sentences


def sentence_truth_score(sentence):
    evidences, _ = retrieve_evidences(sentence, top_k=2)

    # default: neutral → 2.5
    best = 0.5

    for ev in evidences:
        label = fast_nli(sentence, ev)
        if label == 2:          # entailment
            best = 1.0
            break
        elif label == 0:        # contradiction
            best = 0.0

    return best * 5


def score_truthfulness(ans):
    if ans in truth_cache:
        return truth_cache[ans]

    sentences = split_sentences(ans)

    if len(sentences) == 0:
        truth_cache[ans] = 2.5
        return 2.5

    scores = []
    for sent in sentences:
        s = sentence_truth_score(sent)
        scores.append(s)

    final_score = float(np.mean(scores))
    truth_cache[ans] = final_score
    return final_score

def score_relevance(prompt, ans):
    p = embed_model.encode(prompt, convert_to_numpy=True)
    a = embed_model.encode(ans, convert_to_numpy=True)
    sim = np.dot(p, a) / (np.linalg.norm(p)*np.linalg.norm(a) + 1e-8)
    # scale similarity [-1, 1] → [0, 5]
    return float((sim + 1) * 2.5)

UF_PATH = "/content/drive/MyDrive/ultrafeedback_1024_filtered.jsonl"

data = []
with open(UF_PATH, "r", encoding="utf-8") as f:
    for line in f:
        data.append(json.loads(line))

print("UF loaded:", len(data))


def extract_prompt_answers(row):
    prompt   = row["prompt"]
    chosen   = row["chosen"][-1]["content"]
    rejected = row["rejected"][-1]["content"]
    return prompt, chosen, rejected

OUT_PATH = "/content/drive/MyDrive/ultrafb_scored.jsonl"

start = 0
if os.path.exists(OUT_PATH):
    start = sum(1 for _ in open(OUT_PATH, "r", encoding="utf-8"))
    print("Resuming from:", start)

with open(OUT_PATH, "a", encoding="utf-8") as fout:
    for i in tqdm(range(start, len(data)), desc="Scoring UltraFeedback"):
        prompt, chosen, rejected = extract_prompt_answers(data[i])

        t_c = score_truthfulness(chosen)
        t_r = score_truthfulness(rejected)

        r_c = score_relevance(prompt, chosen)
        r_r = score_relevance(prompt, rejected)

        fout.write(json.dumps({
            "prompt": prompt,
            "chosen_answer": chosen,
            "rejected_answer": rejected,
            "chosen_truthfulness": t_c,
            "rejected_truthfulness": t_r,
            "chosen_relevance": r_c,
            "rejected_relevance": r_r
        }, ensure_ascii=False) + "\n")

        if (i+1) % 200 == 0:
            with open(CACHE_PATH, "w", encoding="utf-8") as fcache:
                json.dump(truth_cache, fcache, ensure_ascii=False)
            print("[Checkpoint] Cache updated at row:", i+1)

with open(CACHE_PATH, "w", encoding="utf-8") as fcache:
    json.dump(truth_cache, fcache, ensure_ascii=False)

print("All done! Output saved to:", OUT_PATH)
print("Final truth cache size:", len(truth_cache))


In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
import os
import re
import json
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from torch.optim import AdamW

model_name = "gpt2-large"
output_dir = "./gpt2-large-rm-three-factor"
tokenized_cache_dir = "./rm_tokenized_ultra_v1"

checkpoint_dir = "/content/gdrive/MyDrive/rm-checkpoints/gpt2-large-rm-three-factor"
os.makedirs(checkpoint_dir, exist_ok=True)

save_every_steps = 5000
max_length = 512
batch_size = 1
num_epochs = 1
learning_rate = 2e-6
warmup_ratio = 0.05
logit_l2_weight = 0.01

data_path = "/content/gdrive/MyDrive/ultrafb_scored.jsonl"

print("\n=== Config ===")
print("model:", model_name)
print("dataset:", data_path)
print(f"weights: w_p={w_p}, w_t={w_t}, w_r={w_r}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Using CPU (not recommended for this script)")

def get_latest_checkpoint(dir_path):
    pattern = re.compile(r"checkpoint-step-(\d+)")
    best = None
    max_step = -1

    if not os.path.exists(dir_path):
        return None, None

    for name in os.listdir(dir_path):
        m = pattern.match(name)
        if m:
            step = int(m.group(1))
            if step > max_step:
                max_step = step
                best = os.path.join(dir_path, name)

    return best, max_step

raw_tokenizer = AutoTokenizer.from_pretrained(model_name)

special_tokens = {
    "additional_special_tokens": ["<|prompt|>", "<|assistant|>"]
}
raw_tokenizer.add_special_tokens(special_tokens)

if raw_tokenizer.pad_token is None:
    raw_tokenizer.pad_token = raw_tokenizer.eos_token

resume_ckpt, resume_step = get_latest_checkpoint(checkpoint_dir)

if resume_ckpt:
    print("Resuming from:", resume_ckpt)
    tokenizer = AutoTokenizer.from_pretrained(resume_ckpt)
    model = AutoModelForSequenceClassification.from_pretrained(
        resume_ckpt, num_labels=1
    )
    step = resume_step
else:
    print("Training from scratch")
    tokenizer = raw_tokenizer
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=1
    )
    step = 0

model.resize_token_embeddings(len(tokenizer))
model.to(device)

class UltraFBScoredDataset(Dataset):

    def __init__(self, path, w_p=1.0, w_t=1.0, w_r=1.0):
        self.data = []
        self.w_p = w_p
        self.w_t = w_t
        self.w_r = w_r

        with open(path, encoding="utf-8") as f:
            for line in f:
                row = json.loads(line)

                prompt = row["prompt"]
                chosen = row["chosen_answer"]
                rejected = row["rejected_answer"]

                PrefSignal = 1.0
                TruthAdv = row["chosen_truthfulness"] - row["rejected_truthfulness"]
                RelAdv   = row["chosen_relevance"]     - row["rejected_relevance"]

                combined_sign = (
                    self.w_p * PrefSignal
                    + self.w_t * TruthAdv
                    + self.w_r * RelAdv
                )

                self.data.append({
                    "prompt": prompt,
                    "chosen": chosen,
                    "rejected": rejected,
                    "combined_sign": combined_sign,
                })

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data[idx]

        full_chosen = (
            f"<|prompt|>\n{row['prompt']}\n\n"
            f"<|assistant|>\n{row['chosen']}"
        )
        full_rejected = (
            f"<|prompt|>\n{row['prompt']}\n\n"
            f"<|assistant|>\n{row['rejected']}"
        )

        chosen_tok = tokenizer(
            full_chosen,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt",
        )
        rejected_tok = tokenizer(
            full_rejected,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt",
        )

        return {
            "chosen_input_ids": chosen_tok["input_ids"][0],
            "chosen_attention_mask": chosen_tok["attention_mask"][0],
            "rejected_input_ids": rejected_tok["input_ids"][0],
            "rejected_attention_mask": rejected_tok["attention_mask"][0],
            "combined_sign": torch.tensor(row["combined_sign"], dtype=torch.float),
        }

train_data = UltraFBScoredDataset(
    data_path,
    w_p=w_p,
    w_t=w_t,
    w_r=w_r,
)
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)

print("Dataset size:", len(train_data))

def rm_loss(r_c, r_r, combined_sign, l2_weight=0.01):

    scaled_sign = torch.tanh(combined_sign)

    base = torch.nn.functional.softplus(
        -scaled_sign * (r_c - r_r)
    ).mean()

    l2 = l2_weight * (r_c.pow(2).mean() + r_r.pow(2).mean())

    return base + l2

optimizer = AdamW(model.parameters(), lr=learning_rate)

total_steps = len(train_loader) * num_epochs
warmup_steps = int(warmup_ratio * total_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer, warmup_steps, total_steps
)

print("Total steps:", total_steps)

model.train()

print("\n Start RM Training (three-factor preference) ...\n")

for epoch in range(num_epochs):
    for batch in train_loader:

        optimizer.zero_grad()

        r_c = model(
            input_ids=batch["chosen_input_ids"].to(device),
            attention_mask=batch["chosen_attention_mask"].to(device),
        ).logits.squeeze(-1)

        r_r = model(
            input_ids=batch["rejected_input_ids"].to(device),
            attention_mask=batch["rejected_attention_mask"].to(device),
        ).logits.squeeze(-1)

        combined_sign = batch["combined_sign"].to(device)

        loss = rm_loss(r_c, r_r, combined_sign, logit_l2_weight)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)

        optimizer.step()
        scheduler.step()

        if step % 50 == 0:
            with torch.no_grad():
                scaled_sign = torch.tanh(combined_sign)
                prefer_chosen = (scaled_sign > 0).float()
                prefer_rejected = (scaled_sign < 0).float()


                better_chosen = (r_c > r_r).float()
                better_rejected = (r_c < r_r).float()

                dir_correct = (
                    prefer_chosen * better_chosen
                    + prefer_rejected * better_rejected
                )

                num_dir = (prefer_chosen + prefer_rejected).sum().item()
                if num_dir > 0:
                    dir_acc = dir_correct.sum().item() / num_dir
                else:
                    dir_acc = float("nan")

            print(
                f"Step {step:6d} | Loss={loss.item():.4f} | "
                f"Dir-Acc={dir_acc:.3f}"
            )

        if step % save_every_steps == 0 and step > 0:
            ckpt = os.path.join(checkpoint_dir, f"checkpoint-step-{step}")
            model.save_pretrained(ckpt)
            tokenizer.save_pretrained(ckpt)
            print(f"💾 Saved checkpoint to {ckpt}")

        step += 1

print("\n Training Complete")

os.makedirs(output_dir, exist_ok=True)
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print("Saved final RM to:", output_dir)
